# 🚗 Pothole Detection — YOLOv11 (No Kaggle Key Needed!)
**Method: Upload RDD_SPLIT.zip to Google Drive → Mount in Colab**

---
### 📋 Before Running:
1. Upload `RDD_SPLIT.zip` from your PC to Google Drive (My Drive root)
2. Enable T4 GPU: `Runtime → Change Runtime Type → T4 GPU`
3. Run all cells ▶️

## ✅ Step 1: Check GPU

In [ ]:
!nvidia-smi
import torch
print(f'CUDA: {torch.cuda.is_available()}')
print(f'GPU : {torch.cuda.get_device_name(0)}')

## ✅ Step 2: Install Ultralytics

In [ ]:
!pip install ultralytics -q
print('Done!')

## ✅ Step 3: Mount Google Drive & Extract Dataset
> Make sure `RDD_SPLIT.zip` is uploaded in **My Drive** (root folder)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted!')

In [ ]:
import zipfile, os

ZIP_PATH    = '/content/drive/MyDrive/RDD_SPLIT.zip'
EXTRACT_DIR = '/content/RDD_SPLIT'

if not os.path.exists(ZIP_PATH):
    print('ERROR: RDD_SPLIT.zip not found in Google Drive root!')
    print('Please upload it first from your PC.')
else:
    print('Extracting dataset... (may take 2-3 mins)')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('/content/')
    print('Done!')

# Verify
train_imgs = os.path.join(EXTRACT_DIR, 'train', 'images')
val_imgs   = os.path.join(EXTRACT_DIR, 'val',   'images')
print(f'Train: {len(os.listdir(train_imgs))} images')
print(f'Val  : {len(os.listdir(val_imgs))} images')

## ✅ Step 4: Create YAML Config

In [ ]:
DATASET_PATH = '/content/RDD_SPLIT'
YAML_PATH    = '/content/pothole_data.yaml'

yaml_content = f"""
path: {DATASET_PATH}
train: train/images
val: val/images
test: test/images

names:
  0: Longitudinal Crack
  1: Transverse Crack
  2: Alligator Crack
  3: Other corruption
  4: Pothole
"""

with open(YAML_PATH, 'w') as f:
    f.write(yaml_content.strip())

print('YAML created!')
print(yaml_content)

## ✅ Step 5: Train YOLOv11s 🚀

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11s.pt')

results = model.train(
    data=YAML_PATH,
    epochs=100,
    imgsz=640,
    batch=16,       # T4 = 16GB VRAM, batch 16 is fine
    patience=30,
    save=True,
    device='0',
    amp=True,
    workers=2,
    name='pothole_v1',
    cos_lr=True,
    project='/content/drive/MyDrive/pothole_runs'  # Save directly to Drive!
)

print('Training Complete! Model saved to Google Drive.')

## ✅ Step 6: Download best.pt to your PC

In [ ]:
from google.colab import files
# Model already saved to Drive above
# Also direct download:
files.download('/content/drive/MyDrive/pothole_runs/pothole_v1/weights/best.pt')
print('Downloaded!')

## ✅ Step 7: Quick Inference Test (Optional)

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
import os, random

model = YOLO('/content/drive/MyDrive/pothole_runs/pothole_v1/weights/best.pt')

val_dir = '/content/RDD_SPLIT/val/images'
sample  = random.choice(os.listdir(val_dir))
img_path = os.path.join(val_dir, sample)

results = model.predict(img_path, conf=0.3, save=True, project='/content/test_out')

out_img = f'/content/test_out/predict/{sample}'
display(Image(out_img))